In [6]:
%pip install pandas openpyxl xlrd

Note: you may need to restart the kernel to use updated packages.


In [8]:
#!/usr/bin/env python3
"""
WASDE Feeder Script
====================
Turns a raw monthly USDA WASDE Excel export into a clean, consolidated
"feeder" workbook with 6 tidy tabs, pulled from 8 raw report pages:

    Raw page(s)   ->  Feeder tab
    Page 11       ->  US Wheat
    Page 12       ->  US FeedGrain-Corn
    Page 15       ->  US Soybeans
    Page 18 + 19  ->  World Wheat   (current + projected years merged)
    Page 22 + 23  ->  World Corn    (current + projected years merged)
    Page 28       ->  World Soybean

USAGE
-----
    python wasde_feeder.py path/to/wasde_raw.xls
    python wasde_feeder.py path/to/wasde_raw.xls -o my_output.xlsx

If no output path is given, the script names the file automatically using
the report month/year it finds inside the source file
(e.g. WASDE_Feeder_2026-08.xlsx).

REQUIREMENTS (one-time setup)
------------------------------
    pip install pandas openpyxl xlrd

    - openpyxl is needed to write the output file.
    - xlrd is needed ONLY if USDA's file is the old binary .xls format
      (their site has offered both .xls and .xlsx in different years --
      if you get an error mentioning xlrd, just pip install it).

WHY THIS SCRIPT EXISTS (not just Power Query)
----------------------------------------------
The 8 raw pages use two different table shapes (US commodity-by-year
tables vs. world region-by-year tables), plus quirks: fake "filler"/"Filler"
text in spacer rows, merged-cell blanks that need filling down, and
inconsistent month labels ("Jun" vs "June") within the same page. This
script detects each table's real header row, distinguishes real data from
spacer junk, fills down blank labels, and normalizes month labels -- logic
that's awkward to express in Power Query's UI without hand-written M code.

IF USDA CHANGES THE REPORT LAYOUT
-----------------------------------
USDA occasionally adds/removes rows (e.g. new wheat by-class lines in 2023).
If a future month's file structure shifts enough that this script's output
looks wrong or incomplete, send me (Claude) the new raw file and I'll
update the parsing logic -- then this script will keep working going
forward.
"""

import argparse
import re
import sys
from pathlib import Path

import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, PatternFill
from openpyxl.utils import get_column_letter
from openpyxl.utils.dataframe import dataframe_to_rows

# ---------------------------------------------------------------------------
# Configuration: which raw pages feed which output tab
# ---------------------------------------------------------------------------
US_PAGES = {
    'Page 11': ('US Wheat', 'U.S. Wheat'),
    'Page 12': ('US FeedGrain-Corn', 'U.S. Feed Grains & Corn'),
    'Page 15': ('US Soybeans', 'U.S. Soybeans'),
}
WORLD_PAGE_GROUPS = {
    'World Wheat': ['Page 18', 'Page 19'],
    'World Corn': ['Page 22', 'Page 23'],
    'World Soybean': ['Page 28'],
}

YEAR_RE = re.compile(r'^\d{4}/\d{2}')
MONTH_SET = {'Jun', 'Jul', 'June', 'July'}
MONTH_NORM = {'Jun': 'Jun', 'June': 'Jun', 'Jul': 'Jul', 'July': 'Jul'}
FILLER_SET = {'filler', 'Filler', 'FILLER'}


def is_year_label(v):
    return isinstance(v, str) and YEAR_RE.match(v.strip())


# ---------------------------------------------------------------------------
# Pattern A parser: US commodity-by-year tables (pages 11, 12, 15)
# ---------------------------------------------------------------------------
def parse_us_table(xls_path, sheet):
    df = pd.read_excel(xls_path, sheet_name=sheet, header=None)
    n_rows, n_cols = df.shape
    blocks = []
    i = 0
    current = None
    while i < n_rows:
        row = df.iloc[i]
        year_cols = {}
        for j in range(1, n_cols):
            v = row[j]
            if isinstance(v, str) and (YEAR_RE.match(v.strip()) or 'Proj' in v or 'Est' in v):
                year_cols[j] = v.strip()
        if len(year_cols) >= 2:
            title = row[0] if isinstance(row[0], str) and row[0].strip() else None
            current = {'name': title, 'col_map': dict(year_cols), 'rows': []}
            blocks.append(current)
            if i + 1 < n_rows:
                next_row = df.iloc[i + 1]
                for j in year_cols:
                    v = next_row[j] if j < len(next_row) else None
                    if isinstance(v, str) and v.strip() in MONTH_SET:
                        current['col_map'][j] = current['col_map'][j] + ' (' + MONTH_NORM[v.strip()] + ')'
                i += 1
            i += 1
            continue
        if current is not None:
            label = row[0]
            if isinstance(label, str) and label.strip() and label.strip() not in FILLER_SET:
                if label.strip().lower().startswith('note'):
                    current = None
                    i += 1
                    continue
                vals = {}
                any_val = False
                for j in current['col_map']:
                    v = row[j] if j < len(row) else None
                    if pd.notna(v):
                        vals[j] = v
                        any_val = True
                if any_val:
                    current['rows'].append((label.strip(), vals))
            elif pd.isna(label):
                empty = all(pd.isna(row[j]) for j in current['col_map'] if j < len(row))
                if empty:
                    current = None
        i += 1
    return blocks


def us_blocks_to_df(blocks, page_title):
    frames = []
    for b in blocks:
        if not b['rows']:
            continue
        cols = list(b['col_map'].keys())
        col_names = [b['col_map'][c] for c in cols]
        data = {'Item': [r[0] for r in b['rows']]}
        for c, cname in zip(cols, col_names):
            data[cname] = [r[1].get(c) for r in b['rows']]
        d = pd.DataFrame(data)
        d.insert(0, 'Commodity', b['name'] if b['name'] else page_title)
        frames.append(d)
    if frames:
        return pd.concat(frames, ignore_index=True)
    return pd.DataFrame()


# ---------------------------------------------------------------------------
# Pattern B parser: world region-by-year tables (pages 18, 19, 22, 23, 28)
# ---------------------------------------------------------------------------
def parse_world_table(xls_path, sheet):
    df = pd.read_excel(xls_path, sheet_name=sheet, header=None)
    n_rows, n_cols = df.shape
    records = []
    metric_cols = None
    metric_names = None
    month_col = None
    current_year = None
    last_region = None
    i = 0
    while i < n_rows:
        row = df.iloc[i]
        col0 = row[0]
        if is_year_label(col0):
            current_year = col0.strip()
            metric_cols = []
            metric_names = {}
            month_col = None
            for j in range(1, n_cols):
                v = row[j]
                if isinstance(v, str) and v.strip():
                    metric_cols.append(j)
                    metric_names[j] = v.strip().replace('\n', ' ')
            if 1 not in metric_cols:
                month_col = 1
            last_region = None
            i += 1
            continue
        if current_year is None:
            i += 1
            continue
        if isinstance(col0, str) and (col0.strip().lower().startswith('note') or re.match(r'^\d+/', col0.strip())):
            current_year = None
            i += 1
            continue
        region = col0.strip() if isinstance(col0, str) and col0.strip() else last_region
        month_val = None
        if month_col is not None:
            mv = row[month_col] if month_col < len(row) else None
            if isinstance(mv, str) and mv.strip() in MONTH_SET:
                month_val = MONTH_NORM[mv.strip()]
        vals = {}
        any_val = False
        for j in metric_cols:
            v = row[j] if j < len(row) else None
            if pd.notna(v):
                vals[metric_names[j]] = v
                any_val = True
        if region is not None and any_val:
            rec = {'Marketing Year': current_year, 'Region': region}
            if month_col is not None:
                rec['Report Month'] = month_val
            rec.update(vals)
            records.append(rec)
            last_region = region
        i += 1
    return pd.DataFrame(records)


# ---------------------------------------------------------------------------
# Workbook writer
# ---------------------------------------------------------------------------
def detect_report_label(xls_path):
    """Look at Page 11's top-left cell (e.g. 'July 2026') to name the output file."""
    try:
        df = pd.read_excel(xls_path, sheet_name='Page 11', header=None, nrows=1)
        val = str(df.iloc[0, 0]).strip()
        m = re.match(r'([A-Za-z]+)\s+(\d{4})', val)
        if m:
            month_name, year = m.groups()
            month_num = {
                'January': '01', 'February': '02', 'March': '03', 'April': '04',
                'May': '05', 'June': '06', 'July': '07', 'August': '08',
                'September': '09', 'October': '10', 'November': '11', 'December': '12'
            }.get(month_name, '00')
            return f"{year}-{month_num}"
    except Exception:
        pass
    return "output"


def build_feeder(xls_path, out_path):
    header_font = Font(name='Arial', bold=True, color='FFFFFF')
    header_fill = PatternFill(start_color='1F4E78', end_color='1F4E78', fill_type='solid')
    body_font = Font(name='Arial', size=10)
    title_font = Font(name='Arial', bold=True, size=12)
    note_font = Font(name='Arial', italic=True, size=9, color='808080')

    sheets = {}

    for page, (tab_name, commodity_title) in US_PAGES.items():
        blocks = parse_us_table(xls_path, page)
        sheets[tab_name] = us_blocks_to_df(blocks, commodity_title)

    for tab_name, pages in WORLD_PAGE_GROUPS.items():
        frames = [parse_world_table(xls_path, p) for p in pages]
        sheets[tab_name] = pd.concat(frames, ignore_index=True)

    wb = Workbook()
    wb.remove(wb.active)

    src_name = Path(xls_path).name
    for name, df in sheets.items():
        ws = wb.create_sheet(title=name[:31])
        ws.append([f"Source: {src_name}"])
        ws['A1'].font = note_font
        ws.append([])
        header_row_idx = 3
        for row in dataframe_to_rows(df, index=False, header=True):
            ws.append(row)
        for cell in ws[header_row_idx]:
            cell.font = header_font
            cell.fill = header_fill
            cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        for row in ws.iter_rows(min_row=header_row_idx + 1, max_row=ws.max_row):
            for cell in row:
                cell.font = body_font
                if isinstance(cell.value, (int, float)):
                    cell.number_format = '#,##0.00'
        ws.freeze_panes = ws.cell(row=header_row_idx + 1, column=1).coordinate
        for col_cells in ws.columns:
            max_len = max((len(str(c.value)) for c in col_cells if c.value is not None), default=10)
            col_letter = get_column_letter(col_cells[0].column)
            ws.column_dimensions[col_letter].width = min(max(max_len + 2, 12), 40)

    readme = wb.create_sheet(title='README', index=0)
    readme.column_dimensions['A'].width = 100
    lines = [
        ("WASDE Feeder File", title_font),
        ("", body_font),
        (f"Generated from: {src_name}", body_font),
        ("", body_font),
        ("Tabs in this file:", body_font),
        ("  - US Wheat            (raw Page 11)", body_font),
        ("  - US FeedGrain-Corn   (raw Page 12)", body_font),
        ("  - US Soybeans         (raw Page 15)", body_font),
        ("  - World Wheat         (raw Pages 18 + 19, years merged)", body_font),
        ("  - World Corn          (raw Pages 22 + 23, years merged)", body_font),
        ("  - World Soybean       (raw Page 28)", body_font),
        ("", body_font),
        ("Note: only each page's primary supply & use table was extracted;", body_font),
        ("secondary breakdowns (e.g. wheat 'by class' on Page 11) are not included.", body_font),
        ("", body_font),
        ("If a future month's report layout changes and this script's output looks", body_font),
        ("wrong, send the raw file to Claude to update the parsing logic.", body_font),
    ]
    for i, (text, font) in enumerate(lines, start=1):
        readme.cell(row=i, column=1, value=text).font = font

    wb.save(out_path)


def main():
    parser = argparse.ArgumentParser(description="Clean a raw USDA WASDE Excel export into a feeder workbook.")
    parser.add_argument('input', help="Path to the raw USDA WASDE file (e.g. wasde_08_26_raw.xls)")
    parser.add_argument('-o', '--output', help="Output path (default: WASDE_Feeder_<year>-<month>.xlsx)")
    args = parser.parse_args()

    in_path = Path(args.input)
    if not in_path.exists():
        print(f"ERROR: file not found: {in_path}", file=sys.stderr)
        sys.exit(1)

    if args.output:
        out_path = args.output
    else:
        label = detect_report_label(in_path)
        out_path = f"WASDE_Feeder_{label}.xlsx"

    print(f"Reading {in_path} ...")
    build_feeder(in_path, out_path)
    print(f"Done. Saved: {out_path}")


if __name__ == '__main__':
    main()

usage: ipykernel_launcher.py [-h] [-o OUTPUT] input
ipykernel_launcher.py: error: the following arguments are required: input


SystemExit: 2

In [9]:
in_path = r"C:\Users\JoshA\VS Code Projects\Excel work\wasde 07.26 raw.xls"
out_path = r"C:\Users\JoshA\VS Code Projects\Excel work\WASDE_Feeder_2026-07_v2.xlsx"
build_feeder(in_path, out_path)
print("Done! Saved to:", out_path)

Done! Saved to: C:\Users\JoshA\VS Code Projects\Excel work\WASDE_Feeder_2026-07_v2.xlsx


# just update the final in path and out path for the new raw data file each month and the new name for wasde feeder
